# Piloto: reutilización de continuaciones sintéticas

**IMAC — 8 de noviembre de 2026.** Notebook autónomo: basta subir este archivo a Colab.

Se propone validar la viabilidad del experimento con **OPT-125m + WikiText-2 en inglés**.
Son **64 ejemplos de entrenamiento**, **32 de validación**, **dos ciclos**, proporciones
**0 y 100 %** de tokens objetivo sintéticos y una repetición. Se realizan cinco ajustes:
M0 humano y dos por cada condición. El conjunto de prueba queda reservado.

Los contextos siempre son humanos: «100 %» se refiere solo a continuaciones objetivo.
Cada ajuste vuelve a los pesos preentrenados y crea un optimizador nuevo. La recursión
se transmite por las continuaciones del modelo anterior, no por acumular sus pesos.

### Cómo ejecutarlo

1. En Colab, elegir **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.
   Usar recursos disponibles; este notebook no contrata servicios ni requiere una clave.
2. Ejecutar las celdas en orden. La primera instala las bibliotecas Python necesarias;
   se conserva el PyTorch del entorno. Si Colab pide reiniciar, hacerlo y continuar con imports.
3. El piloto imprime avances y guarda archivos por etapa. Al terminar, descargar el ZIP
   en la última celda. Los archivos en `/content` son temporales: descargar antes de cerrar.

**Estado de entrega:** se comprobaron sintaxis y controles de protocolo localmente.
La instalación, las pruebas PyTorch, las descargas y el entrenamiento requieren ejecución
en Colab y no se presentan como realizados. No modificar parámetros para provocar colapso.
El piloto sirve para decidir escala y tiempos; no se mezcla con los resultados finales.


## 1. Dependencias

Se necesitan Python >=3.11 y PyTorch >=2.6 (normalmente suministrado por Colab).
Transformers carga OPT y el codificador de embeddings; Datasets carga las particiones;
Matplotlib dibuja las trayectorias. Hugging Face Hub, NumPy y Safetensors son dependencias
del mismo entorno. Las versiones efectivas se guardan en el manifiesto.

Las versiones principales de instalación quedan fijadas; las dependencias transitivas
se resuelven al instalar y se registra su versión. No se afirma disponer de un entorno
completamente bloqueado o auditado. Ejecutar desde un runtime limpio.


In [ ]:
import sys
import subprocess
import importlib.util

if sys.version_info < (3, 11):
    raise RuntimeError("Seleccionar un runtime con Python 3.11 o posterior.")
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Falta PyTorch: utilizar un runtime estándar reciente de Colab con GPU.")
if any(name in sys.modules for name in ("transformers", "datasets")):
    raise RuntimeError("Las bibliotecas ya están importadas. Reiniciar la sesión antes de reinstalar.")

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "transformers==5.22.2", "datasets==5.1.0", "matplotlib>=3.9,<4",
                "safetensors>=0.6"], check=True)
print("Instalación terminada. Si Colab solicita reiniciar, hacerlo antes de la siguiente celda.")


## 2. Controles independientes del hardware

Se agrupan los párrafos de un artículo antes de tokenizar. Solo un encabezado de nivel
superior comienza otro artículo; los encabezados se retiran. Las ventanas de 64+128 tokens
no se solapan ni atraviesan artículos. Se conservan IDs y offsets.

Las etiquetas del contexto y del relleno son `-100`. Reemplazar ejemplos completos de
igual longitud garantiza la proporción exacta de tokens objetivo sintéticos. Una semilla
estable se deriva de SHA-256; no depende del hash aleatorio de Python.


In [ ]:
"""Controles del protocolo; solo biblioteca estándar, también embebidos en el notebook."""
import hashlib
import json
import math
import random
import re
from collections import Counter


def stable_seed(*parts):
    raw = json.dumps(parts, ensure_ascii=False, separators=(',', ':')).encode()
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], 'big')


def articles(rows):
    """WikiText: solo un encabezado = título = inicia otro artículo.

    Se retiran encabezados de cualquier nivel; se conservan párrafos y saltos.
    Un fragmento inicial sin título se conserva como artículo propio.
    """
    out, body, title = [], [], 'untitled'
    for row in rows:
        text = row.strip()
        if re.match(r'^=\s+[^=].*?\s+=$', text):
            if body:
                out.append({'article': f'{len(out)}:{title}', 'text': '\n'.join(body)})
            body, title = [], text.strip('= ').strip()
        elif text and not re.match(r'^=+\s.*?\s=+$', text):
            body.append(text)
    if body:
        out.append({'article': f'{len(out)}:{title}', 'text': '\n'.join(body)})
    return out


def windows(docs, context_len, target_len, split):
    width = context_len + target_len
    out = []
    for doc in docs:
        for offset in range(0, len(doc['tokens']) - width + 1, width):
            block = doc['tokens'][offset:offset + width]
            out.append({'id': f"{split}:{doc['article']}:{offset}",
                        'split': split, 'article': doc['article'], 'offset': offset,
                        'context': block[:context_len], 'target': block[context_len:]})
    return out


def labelled(context, target, pad_id, length=None):
    if not context or not target:
        raise ValueError('Contexto y objetivo deben ser no vacíos.')
    ids = list(context) + list(target)
    length = len(ids) if length is None else length
    if length < len(ids):
        raise ValueError('El relleno no puede truncar el objetivo.')
    padding = length - len(ids)
    return (ids + [pad_id] * padding,
            [1] * len(ids) + [0] * padding,
            [-100] * len(context) + list(target) + [-100] * padding)


def mix_targets(human, synthetic, alpha, seed):
    """Reemplaza ejemplos completos; longitud fija implica proporción exacta de tokens."""
    if not human or not 0 <= alpha <= 1:
        raise ValueError('Corpus vacío o proporción inválida.')
    count = len(human) * alpha
    if not math.isclose(count, round(count)):
        raise ValueError('La proporción debe dar un número entero de ejemplos.')
    length = len(human[0]['target'])
    if any(len(x['target']) != length for x in human):
        raise ValueError('Objetivos humanos de longitud desigual.')
    if alpha and (synthetic is None or len(synthetic) != len(human)
                  or any(len(x) != length for x in synthetic)):
        raise ValueError('Objetivos sintéticos incompletos o de longitud desigual.')
    chosen = set(random.Random(seed).sample(range(len(human)), round(count)))
    return [dict(x, context=list(x['context']),
                 target=list(synthetic[i] if i in chosen else x['target']),
                 synthetic=i in chosen) for i, x in enumerate(human)]


WORD_PATTERN = r"\w+(?:['’]\w+)*"


def words(text):
    return re.findall(WORD_PATTERN, text.lower(), flags=re.UNICODE)


def lexical_metrics(texts, limit):
    if not texts or limit < 4:
        raise ValueError('Se requieren textos y al menos cuatro palabras por texto.')
    seqs = [words(t) for t in texts]
    short = sum(len(s) < limit for s in seqs)
    result = {'word_limit': limit, 'n_texts': len(texts), 'short_texts': short,
              'distinct_2': None, 'distinct_3': None, 'repeat_4_mean': None}
    if short:
        return result
    seqs = [s[:limit] for s in seqs]
    for n in (2, 3):
        grams = [tuple(s[i:i+n]) for s in seqs for i in range(len(s)-n+1)]
        result[f'distinct_{n}'] = len(set(grams)) / len(grams)
    repeats = []
    for s in seqs:
        grams = [tuple(s[i:i+4]) for i in range(len(s)-3)]
        repeats.append(1 - len(set(grams)) / len(grams))
    result['repeat_4_mean'] = sum(repeats) / len(repeats)
    return result


In [ ]:
"""Controles offline: ejecutar con python -m unittest discover -s Experimentos -v."""
import unittest



class ProtocolTests(unittest.TestCase):
    def test_article_boundaries_and_subheadings(self):
        rows = [' = One = \n', 'alpha', ' == Sub == \n', 'beta', ' = Two = \n', 'gamma']
        docs = articles(rows)
        self.assertEqual(len(docs), 2)
        self.assertIn('beta', docs[0]['text'])
        self.assertNotIn('gamma', docs[0]['text'])
        self.assertNotIn('One', docs[0]['text'])

    def test_no_cross_article_or_overlapping_windows(self):
        docs = [{'article': 'a', 'tokens': list(range(13))},
                {'article': 'b', 'tokens': list(range(100, 105))}]
        blocks = windows(docs, 2, 3, 'train')
        self.assertEqual([x['offset'] for x in blocks], [0, 5, 0])
        self.assertEqual(blocks[1]['context'] + blocks[1]['target'], list(range(5, 10)))
        self.assertEqual(blocks[2]['target'], [102, 103, 104])
        self.assertEqual(len({x['id'] for x in blocks}), 3)

    def test_context_and_padding_are_not_training_targets(self):
        ids, mask, labels = labelled([8, 9], [10, 11, 12], pad_id=1, length=7)
        self.assertEqual(ids, [8, 9, 10, 11, 12, 1, 1])
        self.assertEqual(mask, [1, 1, 1, 1, 1, 0, 0])
        self.assertEqual(labels, [-100, -100, 10, 11, 12, -100, -100])

    def test_mixture_is_exact_and_keeps_human_context(self):
        human = [{'id': str(i), 'context': [i], 'target': [7, 7, 7, 7]} for i in range(4)]
        synthetic = [[9, 9, 9, 9] for _ in human]
        out = mix_targets(human, synthetic, .5, 42)
        self.assertEqual(sum(x['synthetic'] for x in out), 2)
        self.assertEqual(sum(x['target'].count(9) for x in out), 8)
        self.assertEqual([x['context'] for x in out], [[i] for i in range(4)])
        self.assertEqual(human[0]['target'], [7, 7, 7, 7])
        with self.assertRaises(ValueError):
            mix_targets(human, [[9]] * 4, 1, 42)
        with self.assertRaises(ValueError):
            mix_targets(human, synthetic, .3, 42)

    def test_seeds_stable_and_namespaces_separate(self):
        self.assertEqual(stable_seed(42, 'train', 1), stable_seed(42, 'train', 1))
        self.assertNotEqual(stable_seed(42, 'train', 1), stable_seed(42, 'eval', 1))

    def test_lexical_metrics_do_not_cross_text_boundaries(self):
        m = lexical_metrics(['A b a b a b', 'c d c d c d'], 6)
        self.assertAlmostEqual(m['distinct_2'], 4 / 10)
        self.assertAlmostEqual(m['repeat_4_mean'], 1 / 3)
        self.assertEqual(m['short_texts'], 0)

    def test_short_texts_are_not_silently_dropped(self):
        m = lexical_metrics(['a b c d', 'a'], 4)
        self.assertEqual(m['short_texts'], 1)
        self.assertIsNone(m['distinct_2'])
        self.assertIsNone(m['repeat_4_mean'])



suite = unittest.defaultTestLoader.loadTestsFromTestCase(ProtocolTests)
report = unittest.TextTestRunner(verbosity=2).run(suite)
assert report.wasSuccessful(), "No continuar: fallaron controles del protocolo."


## 3. Imports y parámetros del piloto

Los valores se mantienen como propuesta operativa. `allow_cpu=False` detiene la ejecución
sin GPU; cambiarlo solo si se desea deliberadamente usar CPU. `save_checkpoints=False`
evita guardar varios gigabytes de pesos: sí se guardan datos, métricas, semillas,
revisiones, hashes y tiempos. Puede activarse antes de ejecutar si hay espacio.

El ajuste es completo, una época, float32, AdamW (lr 2e-5, betas .9/.999,
eps 1e-8, weight decay .01 en todos los parámetros), clipping 1, sin scheduler.
Lote efectivo 8 mediante microbatches de 2: ocho actualizaciones por ajuste del piloto.
Se comprueba el hash de todos los pesos al reiniciar cada ajuste.


In [ ]:
import csv
import importlib.metadata
import os
import platform
import shutil
import sys
import time
import uuid
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

# No se necesitan tokens para estos repositorios públicos.
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, OPTConfig, OPTForCausalLM
import matplotlib.pyplot as plt


@dataclass(frozen=True)
class Config:
    model: str = 'facebook/opt-125m'
    dataset: str = 'Salesforce/wikitext'
    subset: str = 'wikitext-2-raw-v1'
    embedding: str = 'sentence-transformers/all-MiniLM-L6-v2'
    seed: int = 20261108
    train_examples: int = 64
    eval_examples: int = 32
    context_tokens: int = 64
    target_tokens: int = 128
    cycles: int = 2
    alphas: tuple = (0.0, 1.0)
    micro_batch: int = 2
    effective_batch: int = 8
    sample_batch: int = 8
    learning_rate: float = 2e-5
    betas: tuple = (0.9, 0.999)
    epsilon: float = 1e-8
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    temperature: float = 1.0
    top_p: float = 0.95
    max_words: int = 32
    embedding_max_tokens: int = 256
    allow_cpu: bool = False
    save_checkpoints: bool = False


CFG = Config()


def validate_config(cfg):
    if (cfg.train_examples != 64 or cfg.eval_examples != 32 or cfg.cycles != 2
            or tuple(cfg.alphas) != (0.0, 1.0)):
        raise ValueError('Este notebook está acotado al piloto; no activar el estudio final aquí.')
    if cfg.train_examples % cfg.effective_batch or cfg.effective_batch % cfg.micro_batch:
        raise ValueError('Los lotes deben dividir el corpus y el lote efectivo exactamente.')
    if not 0 < cfg.top_p <= 1 or cfg.temperature <= 0:
        raise ValueError('Parámetros de muestreo inválidos.')


def seed_training(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def weight_hash(model):
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def reset_for_fit(model, base_state, expected_hash, cfg, seed):
    model.load_state_dict(base_state, strict=True)
    model.zero_grad(set_to_none=True)
    observed_hash = weight_hash(model)
    if observed_hash != expected_hash:
        raise RuntimeError('La inicialización no coincide con el modelo base.')
    seed_training(seed)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, betas=cfg.betas,
        eps=cfg.epsilon, weight_decay=cfg.weight_decay, foreach=False)
    if len(optimizer.state):
        raise RuntimeError('El optimizador no empieza vacío.')
    return optimizer, observed_hash


def batch_tensors(rows, pad_id, device):
    packed = [labelled(x['context'], x['target'], pad_id) for x in rows]
    return {name: torch.tensor([x[i] for x in packed], dtype=torch.long, device=device)
            for i, name in enumerate(('input_ids', 'attention_mask', 'labels'))}


def fit_one(model, base_state, base_hash, rows, cfg, seed, device, pad_id):
    optimizer, initial_hash = reset_for_fit(model, base_state, base_hash, cfg, seed)
    generator = torch.Generator().manual_seed(stable_seed(seed, 'shuffle'))
    loader = DataLoader(rows, batch_size=cfg.micro_batch, shuffle=True,
                        generator=generator, collate_fn=list, num_workers=0)
    accumulation = cfg.effective_batch // cfg.micro_batch
    model.train()
    losses, updates, seen = [], 0, 0
    for step, batch in enumerate(loader, start=1):
        data = batch_tensors(batch, pad_id, device)
        labels = data['labels']
        assert (labels[:, :cfg.context_tokens] == -100).all().item()
        assert (labels[:, cfg.context_tokens:] != -100).sum().item() == len(batch) * cfg.target_tokens
        output = model(**data, use_cache=False)
        if not torch.isfinite(output.loss).item():
            raise RuntimeError('Pérdida no finita; no se cambian parámetros automáticamente.')
        losses.append(float(output.loss.detach().cpu()))
        (output.loss / accumulation).backward()
        seen += len(batch) * cfg.target_tokens
        if step % accumulation == 0:
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            if not torch.isfinite(norm).item():
                raise RuntimeError('Gradiente no finito.')
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            updates += 1
    assert updates == len(rows) // cfg.effective_batch
    assert seen == len(rows) * cfg.target_tokens
    del optimizer
    model.eval()
    return {'initial_weight_sha256': initial_hash, 'train_seed': seed,
            'optimizer_initial_entries': 0, 'updates': updates,
            'target_tokens_seen': seen, 'train_loss_mean': sum(losses)/len(losses),
            'final_weight_sha256': weight_hash(model)}


## 4. Generación y comprobaciones PyTorch

Muestreo autorregresivo con caché, temperatura 1 y top-p .95; sin top-k, penalización de
repetición ni beam search. **Propuesta de política EOS del piloto:** se excluyen todos los
IDs especiales del tokenizador, incluido EOS, para obtener exactamente 128 tokens de texto.
Esta restricción modifica la distribución de generación y debe declararse en el estudio;
su conveniencia sigue pendiente de revisión tras el piloto. No se rechazan textos por calidad.

Cada solicitud tiene su generador independiente. La evaluación usa las mismas semillas
por prompt en todas las etapas (comparación pareada). La generación de datos de entrenamiento
utiliza otra familia de semillas por condición/ciclo; el ajuste y el barajado usan otras.

Antes de descargar el modelo se probarán pérdida causal, relleno, reinicio y muestreo con
un OPT diminuto de pesos aleatorios. No son resultados científicos ni requieren descargas.


In [ ]:
def nucleus_draw(logits, generators, cfg, forbidden_ids):
    logits = logits.float().clone() / cfg.temperature
    logits[:, forbidden_ids] = -float('inf')
    values, indices = torch.sort(logits, descending=True, dim=-1)
    probs = torch.softmax(values, dim=-1)
    remove = (torch.cumsum(probs, dim=-1) - probs) >= cfg.top_p
    values[remove] = -float('inf')
    probs = torch.softmax(values, dim=-1)
    draws = [torch.multinomial(p, 1, generator=g) for p, g in zip(probs, generators)]
    return indices.gather(1, torch.stack(draws))


@torch.inference_mode()
def sample_targets(model, rows, cfg, device, forbidden_ids, namespace, progress=True):
    """Un RNG por solicitud; no se hereda el RNG de entrenamiento.

    Política piloto: longitud fija y ningún token especial (incluye EOS/PAD/BOS).
    Se conserva el token que cruza el umbral top-p. Sin top-k ni penalizaciones.
    """
    model.eval()
    targets, seeds = [], []
    for start in range(0, len(rows), cfg.sample_batch):
        batch = rows[start:start+cfg.sample_batch]
        batch_seeds = [stable_seed(cfg.seed, 'sampling', namespace, x['id']) for x in batch]
        generators = [torch.Generator(device=device).manual_seed(s) for s in batch_seeds]
        ids = torch.tensor([x['context'] for x in batch], dtype=torch.long, device=device)
        generated, cache = [], None
        attention = torch.ones_like(ids)
        for _ in range(cfg.target_tokens):
            output = model(input_ids=ids, attention_mask=attention,
                           past_key_values=cache, use_cache=True)
            ids = nucleus_draw(output.logits[:, -1, :], generators, cfg, forbidden_ids)
            generated.append(ids)
            cache = output.past_key_values
            attention = torch.cat([attention, torch.ones_like(ids)], dim=1)
        result = torch.cat(generated, dim=1).cpu().tolist()
        assert all(len(x) == cfg.target_tokens for x in result)
        assert all(not (set(x) & set(forbidden_ids)) for x in result)
        targets.extend(result)
        seeds.extend(batch_seeds)
        if progress:
            print(f'  Muestreo {min(start+cfg.sample_batch, len(rows))}/{len(rows)}', flush=True)
    return targets, seeds


def torch_smoke_checks():
    """Sin descargas: prueba real de pérdida causal, caché, semillas y reinicio."""
    from dataclasses import replace
    seed_training(17)
    toy = OPTForCausalLM(OPTConfig(vocab_size=32, hidden_size=16, ffn_dim=32,
        word_embed_proj_dim=16, num_hidden_layers=1, num_attention_heads=2,
        max_position_embeddings=32, dropout=0.0, attention_dropout=0.0,
        pad_token_id=1, bos_token_id=2, eos_token_id=2)).cpu()
    config = replace(CFG, context_tokens=2, target_tokens=4,
                     micro_batch=1, effective_batch=2, sample_batch=2)
    rows = [{'id': str(i), 'context': [4, 5], 'target': [6, 7, 8, 9]} for i in range(2)]
    data = batch_tensors(rows, 1, 'cpu')
    toy.eval()
    output = toy(**data, use_cache=False)
    manual = F.cross_entropy(output.logits[:, :-1, :].reshape(-1, 32),
                             data['labels'][:, 1:].reshape(-1), ignore_index=-100)
    assert torch.allclose(output.loss, manual, atol=1e-6)
    assert data['labels'][:, 1:].ne(-100).sum().item() == 8
    a, _ = sample_targets(toy, rows, config, 'cpu', [0, 1, 2, 3], 'check', False)
    torch.rand(123)  # Perturbar RNG global no debe cambiar la generación.
    b, _ = sample_targets(toy, rows, config, 'cpu', [0, 1, 2, 3], 'check', False)
    assert a == b
    base, signature = cpu_state(toy), weight_hash(toy)
    fit_one(toy, base, signature, rows, config, 99, 'cpu', 1)
    assert weight_hash(toy) != signature
    optimizer, observed = reset_for_fit(toy, base, signature, config, 99)
    assert observed == signature and len(optimizer.state) == 0
    # El relleno debe estar excluido también después del desplazamiento causal.
    ids, mask, labels = labelled([4, 5], [6, 7], 1, 7)
    padded = {k: torch.tensor([v]) for k, v in
              zip(('input_ids', 'attention_mask', 'labels'), (ids, mask, labels))}
    toy.eval()
    output = toy(**padded, use_cache=False)
    loss = F.cross_entropy(output.logits[:, :-1].reshape(-1, 32),
                           padded['labels'][:, 1:].reshape(-1), ignore_index=-100)
    assert torch.allclose(output.loss, loss, atol=1e-6)
    del optimizer, toy
    print('Controles PyTorch: pérdida causal, relleno, muestreo y reinicio correctos.')


## 5. Preparación y trazabilidad

Se resuelven y registran los commits de modelo, dataset y codificador. Los repositorios
se cargan sin ejecutar código remoto ni usar tokens de autenticación. Se prefieren pesos
Safetensors; si OPT solo ofrece `.bin`, se exige la carga restringida `weights_only=True`
con PyTorch >=2.6. No se usa deserialización de objetos arbitrarios.

Solo se solicitan `train` y `validation`; no se carga ni se evalúa `test`.
Se muestrean bloques sin reemplazo de cada partición y se comprueban duplicados exactos
entre los bloques seleccionados. Esta comprobación no demuestra ausencia de toda posible
contaminación previa del preentrenamiento ni de similitud entre artículos.

Cada ejecución crea una carpeta nueva. En caso de error conserva resultados parciales;
una nueva ejecución empieza otro piloto. No hay reanudación automática de checkpoints.


In [ ]:
def write_json(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)


def write_jsonl(path, rows):
    with Path(path).open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')


def hash_json(obj):
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True).encode()).hexdigest()


class Clock:
    def __init__(self, path, device):
        self.path, self.device, self.records = path, device, []

    @contextmanager
    def measure(self, stage, **tags):
        if self.device == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        status = 'error'
        try:
            yield
            status = 'ok'
        finally:
            if self.device == 'cuda':
                torch.cuda.synchronize()
            self.records.append(dict(stage=stage, seconds=time.perf_counter()-start,
                                     status=status, **tags))
            write_json(self.path, self.records)


def select_data(cfg, tokenizer, revision):
    selected, fingerprints = {}, {}
    # Se solicitan solo train y validation; NO se carga ni se evalúa test.
    for split, count in [('train', cfg.train_examples), ('validation', cfg.eval_examples)]:
        raw = load_dataset(cfg.dataset, cfg.subset, split=split, revision=revision, token=False)
        docs = articles(raw['text'])
        for doc in docs:
            doc['tokens'] = tokenizer(doc['text'], add_special_tokens=False,
                                      truncation=False)['input_ids']
        pool = windows(docs, cfg.context_tokens, cfg.target_tokens, split)
        if len(pool) < count:
            raise RuntimeError(f'{split}: solo hay {len(pool)} bloques; se necesitan {count}.')
        chosen = random.Random(stable_seed(cfg.seed, 'data', split)).sample(pool, count)
        selected[split] = chosen
        fingerprints[split] = {'dataset_fingerprint': raw._fingerprint,
                               'available_windows': len(pool), 'selected_sha256': hash_json(chosen)}
    # Control de duplicados exactos seleccionados entre particiones, sin sustituirlos a posteriori.
    train_blocks = {tuple(x['context'] + x['target']) for x in selected['train']}
    if any(tuple(x['context'] + x['target']) in train_blocks for x in selected['validation']):
        raise RuntimeError('Hay bloques idénticos seleccionados entre train y validation; revisar corpus.')
    return selected, fingerprints


## 6. Evaluación

- **Perplejidad:** pérdida causal sobre las 128 posiciones humanas reservadas de validación;
  incluye la predicción del primer objetivo desde el contexto. No se evalúa sobre entrenamiento.
- **Distinct-2/3:** n-gramas únicos / n-gramas totales del corpus; sin cruzar fronteras de textos.
- **Repetición de 4-gramas:** promedio por texto de `1 - únicos / total`.
- **Dispersión semántica:** promedio de `1 - coseno` sobre todos los pares de textos, usando
  MiniLM, mean pooling con máscara y normalización L2. Mide dispersión, no calidad ni fidelidad.

Se fija W = mínimo entre 32 palabras y la continuación humana más corta de validación,
antes de generar. Métricas léxicas y semánticas usan las primeras W palabras en minúsculas.
Si alguna generación no llega a W, se informa su número y las métricas comparativas quedan
ausentes para toda esa etapa; no se filtran ejemplos ni se rellena texto. PPL sigue disponible.
El codificador tiene límite de 256 wordpieces; se registra cualquier truncamiento.


In [ ]:
@torch.inference_mode()
def human_perplexity(model, rows, cfg, device, pad_id):
    model.eval()
    nll, count = 0.0, 0
    for start in range(0, len(rows), cfg.micro_batch):
        batch = batch_tensors(rows[start:start+cfg.micro_batch], pad_id, device)
        output = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], use_cache=False)
        labels = batch['labels'][:, 1:].contiguous()
        loss = F.cross_entropy(output.logits[:, :-1, :].float().reshape(-1, output.logits.size(-1)),
                               labels.reshape(-1), reduction='sum', ignore_index=-100)
        nll += float(loss.cpu())
        count += int(labels.ne(-100).sum().item())
    assert count == len(rows) * cfg.target_tokens
    mean = nll / count
    if not math.isfinite(mean) or mean > 700:
        raise RuntimeError('Perplejidad no finita o fuera de rango.')
    return {'human_nll': mean, 'human_perplexity': math.exp(mean), 'ppl_target_tokens': count}


@torch.inference_mode()
def semantic_dispersion(texts, encoder, encoder_tokenizer, cfg, limit):
    # Mismas primeras W palabras que en diversidad léxica; sin descartar textos cortos.
    if any(len(words(t)) < limit for t in texts):
        return {'semantic_dispersion': None, 'embedding_truncated_texts': None}
    embeddings, truncated = [], 0
    encoder.eval()
    for start in range(0, len(texts), 16):
        batch = [' '.join(words(t)[:limit]) for t in texts[start:start+16]]
        lengths = [len(x) for x in encoder_tokenizer(batch, truncation=False)['input_ids']]
        truncated += sum(n > cfg.embedding_max_tokens for n in lengths)
        inputs = encoder_tokenizer(batch, padding=True, truncation=True,
                                   max_length=cfg.embedding_max_tokens, return_tensors='pt')
        output = encoder(**inputs).last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).to(output.dtype)
        pooled = (output * mask).sum(1) / mask.sum(1).clamp_min(1)
        embeddings.append(F.normalize(pooled, p=2, dim=1))
    matrix = torch.cat(embeddings)
    similarity = matrix @ matrix.T
    upper = torch.triu_indices(len(texts), len(texts), offset=1)
    dispersion = 1 - similarity[upper[0], upper[1]].mean().item()
    return {'semantic_dispersion': dispersion, 'embedding_truncated_texts': truncated}


def evaluate_stage(model, valid, tokenizer, encoder, enc_tokenizer, cfg,
                   device, forbidden, limit, folder, clock):
    with clock.measure('eval_generation', stage_id=folder.name):
        generated, seeds = sample_targets(model, valid, cfg, device, forbidden, 'validation_paired')
    texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
    with clock.measure('eval_perplexity', stage_id=folder.name):
        metrics = human_perplexity(model, valid, cfg, device, tokenizer.pad_token_id)
    metrics.update(lexical_metrics(texts, limit))
    with clock.measure('eval_embeddings', stage_id=folder.name):
        metrics.update(semantic_dispersion(texts, encoder, enc_tokenizer, cfg, limit))
    with clock.measure('write_outputs', stage_id=folder.name):
        write_jsonl(folder/'validation_generations.jsonl', [
            {'id': row['id'], 'seed': seed, 'target_ids': ids, 'text': text}
            for row, seed, ids, text in zip(valid, seeds, generated, texts)])
        write_json(folder/'metrics.json', metrics)
    return metrics


## 7. Ejecutar los cinco ajustes

La siguiente celda define el procedimiento. La celda posterior lo ejecuta: carga recursos,
ajusta M0 y construye las dos cadenas. Cada ajuste inicia en el checkpoint base y optimizador
nuevo. M0 es compartido: la primera generación sintética de ambas condiciones parte del mismo
punto. El control 0 % usa siempre los objetivos humanos originales.

Se cronometran carga, ajuste, generación de corpus, generación de validación, perplejidad,
embeddings y escritura por separado. No se promete una duración antes de medirla.


In [ ]:
def run_pilot(cfg=CFG):
    validate_config(cfg)
    if sys.version_info < (3, 11):
        raise RuntimeError('Se requiere Python 3.11 o posterior.')
    major_minor = tuple(int(n) for n in torch.__version__.split('+')[0].split('.')[:2])
    if major_minor < (2, 6):
        raise RuntimeError('Se requiere PyTorch >=2.6. Usar un runtime reciente de Colab.')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('PyTorch:', torch.__version__, '| Dispositivo:', device, flush=True)
    if device == 'cpu' and not cfg.allow_cpu:
        raise RuntimeError('No hay GPU CUDA. Elegir GPU en Colab y ejecutar desde el principio. '
                           'Para CPU deliberada, cambiar allow_cpu=True; no se promete duración.')
    torch.use_deterministic_algorithms(True)
    if device == 'cuda':
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        print('GPU:', torch.cuda.get_device_name(0), flush=True)
    torch_smoke_checks()
    root = Path('/content/imac_piloto') if Path('/content').is_dir() else Path.cwd()/'salidas_piloto'
    run = root/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
    run.mkdir(parents=True, exist_ok=False)
    clock = Clock(run/'timings.json', device)
    # Solo nombres/versiones; no pip freeze con posibles URLs privadas o credenciales.
    versions = dict(sorted((dist.metadata['Name'], dist.version)
                           for dist in importlib.metadata.distributions()
                           if dist.metadata.get('Name')))
    manifest = {'status': 'running', 'kind': 'pilot_not_final_results', 'config': asdict(cfg),
                'python': platform.python_version(), 'versions': versions, 'device': device,
                'started_utc': datetime.now(timezone.utc).isoformat(), 'test_used': False,
                'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else None,
                'gpu_bytes': torch.cuda.get_device_properties(0).total_memory if device == 'cuda' else None,
                'eos_policy': '128 tokens; todos los IDs especiales excluidos; sin rechazo ni filtrado',
                'seed_policy': 'SHA256; entrenamiento separado; evaluación pareada entre etapas',
                'optimizer': 'AdamW; una época; sin scheduler; decay uniforme en parámetros',
                'precision': 'float32; sin AMP ni TF32; determinismo estricto en este entorno',
                'checkpoints_saved': cfg.save_checkpoints,
                'resume_policy': 'no reanudación automática; cada ejecución crea otra carpeta'}
    write_json(run/'manifest.json', manifest)
    print('Directorio de salida:', run, flush=True)
    try:
        api = HfApi(token=False)
        with clock.measure('resolve_revisions'):
            model_info = api.model_info(cfg.model)
            dataset_info = api.dataset_info(cfg.dataset)
            embed_info = api.model_info(cfg.embedding)
            revisions = {'model': model_info.sha, 'dataset': dataset_info.sha, 'embedding': embed_info.sha}
            safe_model = any(x.rfilename.endswith('.safetensors') for x in model_info.siblings)
        manifest.update(revisions=revisions, model_uses_safetensors=safe_model)
        write_json(run/'manifest.json', manifest)
        with clock.measure('load_tokenizer_data'):
            tokenizer = AutoTokenizer.from_pretrained(cfg.model, revision=revisions['model'],
                                                      trust_remote_code=False, token=False)
            selected, fingerprints = select_data(cfg, tokenizer, revisions['dataset'])
            human, valid = selected['train'], selected['validation']
            references = tokenizer.batch_decode([r['target'] for r in valid], skip_special_tokens=True)
            limit = min(cfg.max_words, min(len(words(t)) for t in references))
            if limit < 4:
                raise RuntimeError('La referencia no permite un límite léxico común >=4; revisar antes de continuar.')
            manifest.update(data=fingerprints, word_limit=limit, word_regex=WORD_PATTERN,
                            forbidden_generation_ids=tokenizer.all_special_ids)
            write_json(run/'manifest.json', manifest)
            write_jsonl(run/'train_human.jsonl', human)
            write_jsonl(run/'validation_human.jsonl', valid)
        with clock.measure('load_model'):
            model = AutoModelForCausalLM.from_pretrained(cfg.model, revision=revisions['model'],
                trust_remote_code=False, token=False, use_safetensors=safe_model,
                weights_only=True, dtype=torch.float32).to(device)
            if not all(p.requires_grad for p in model.parameters()):
                raise RuntimeError('No están habilitados todos los parámetros.')
            base = cpu_state(model)
            base_hash = weight_hash(model)
            manifest['base_weight_sha256'] = base_hash
            manifest['parameters'] = sum(p.numel() for p in model.parameters())
            write_json(run/'manifest.json', manifest)
        with clock.measure('load_embeddings'):
            enc_tokenizer = AutoTokenizer.from_pretrained(cfg.embedding, revision=revisions['embedding'],
                                                          trust_remote_code=False, token=False)
            encoder = AutoModel.from_pretrained(cfg.embedding, revision=revisions['embedding'],
                trust_remote_code=False, token=False, use_safetensors=True,
                dtype=torch.float32).cpu().eval()
        with clock.measure('reference_embeddings'):
            reference_metrics = lexical_metrics(references, limit)
            reference_metrics.update(semantic_dispersion(references, encoder, enc_tokenizer, cfg, limit))
        write_json(run/'human_reference_metrics.json', reference_metrics)
        results = []

        def stage(name, rows, cycle, alpha):
            folder = run/name
            folder.mkdir(exist_ok=False)
            write_jsonl(folder/'training_corpus.jsonl', rows)
            # Misma semilla de ajuste para ambas proporciones en el mismo ciclo.
            seed = stable_seed(cfg.seed, 'fit', 'replicate_0', cycle)
            with clock.measure('fit', stage_id=name):
                fit_record = fit_one(model, base, base_hash, rows, cfg, seed, device, tokenizer.pad_token_id)
            write_json(folder/'fit.json', fit_record)
            if cfg.save_checkpoints:
                with clock.measure('save_checkpoint', stage_id=name):
                    model.save_pretrained(folder/'checkpoint', safe_serialization=True)
                    tokenizer.save_pretrained(folder/'checkpoint')
            metrics = evaluate_stage(model, valid, tokenizer, encoder, enc_tokenizer, cfg,
                device, tokenizer.all_special_ids, limit, folder, clock)
            results.append(dict(stage=name, cycle=cycle, alpha=alpha, **metrics))
            write_json(run/'results.json', results)
            with (run/'results.csv').open('w', newline='', encoding='utf-8') as stream:
                writer = csv.DictWriter(stream, fieldnames=list(results[0]))
                writer.writeheader()
                writer.writerows(results)
            print(f'{name}: PPL={metrics["human_perplexity"]:.3f}; '
                  f'Distinct-2={metrics["distinct_2"]}; cortos={metrics["short_texts"]}', flush=True)

        stage('M0_human', human, 0, None)
        initial = cpu_state(model)
        for alpha in cfg.alphas:
            model.load_state_dict(initial, strict=True)
            for cycle in range(1, cfg.cycles+1):
                name = f'alpha_{int(alpha*100):03d}_g{cycle}'
                synthetic, sample_seeds = None, []
                if alpha:
                    with clock.measure('train_generation', stage_id=name):
                        synthetic, sample_seeds = sample_targets(model, human, cfg, device,
                            tokenizer.all_special_ids, f'train_alpha_{alpha}_cycle_{cycle}')
                    write_jsonl(run/f'{name}_synthetic.jsonl', [
                        {'id': x['id'], 'seed': s, 'target_ids': ids}
                        for x, s, ids in zip(human, sample_seeds, synthetic)])
                mixture_seed = stable_seed(cfg.seed, 'mixture', alpha, cycle)
                rows = mix_targets(human, synthetic, alpha, mixture_seed)
                assert sum(x['synthetic'] for x in rows) * cfg.target_tokens == int(alpha * len(rows) * cfg.target_tokens)
                stage(name, rows, cycle, alpha)
        assert len(results) == 5
        manifest.update(status='complete', finished_utc=datetime.now(timezone.utc).isoformat(),
                        fit_count=len(results), test_used=False)
        write_json(run/'manifest.json', manifest)
        print('Piloto terminado. Son controles de viabilidad, no resultados del estudio final.', flush=True)
        return run, results
    except Exception as error:
        # No volcar variables de entorno, credenciales ni encabezados de red.
        manifest.update(status='interrupted_or_failed', error_type=type(error).__name__)
        write_json(run/'manifest.json', manifest)
        print('Ejecución detenida; se conservan archivos parciales en', run, flush=True)
        raise


In [ ]:
RUN_DIR, RESULTS = run_pilot(CFG)


## 8. Gráficas y descarga

Las trayectorias son descriptivas: solo hay una repetición por condición y dos ciclos.
No se calculan p-valores ni se demuestra colapso a partir de estas gráficas.
El ZIP incluye manifiesto, corpus, muestras, semillas, métricas, registros de ajustes,
tiempos, CSV y figura. Los checkpoints solo se incluyen si se activaron previamente.


In [ ]:
def summarize_pilot(run, results):
    columns = ['distinct_2', 'distinct_3', 'repeat_4_mean', 'semantic_dispersion', 'human_perplexity']
    fig, axes = plt.subplots(1, len(columns), figsize=(19, 3.6))
    for ax, metric in zip(axes, columns):
        for alpha in CFG.alphas:
            points = [results[0]] + [r for r in results[1:] if r['alpha'] == alpha]
            ys = [r[metric] if r[metric] is not None else float('nan') for r in points]
            ax.plot([r['cycle'] for r in points], ys, marker='o', label=f'{alpha:.0%}')
        ax.set_title(metric)
        ax.set_xlabel('Ciclo')
        ax.set_xticks([0, 1, 2])
        ax.grid(alpha=.3)
    axes[0].legend(title='Objetivos sintéticos')
    fig.suptitle('Piloto: una cadena por condición; sin inferencia estadística')
    fig.tight_layout()
    fig.savefig(run/'pilot_metrics.png', dpi=160, bbox_inches='tight')
    plt.show()
    times = json.loads((run/'timings.json').read_text(encoding='utf-8'))
    print('\nTiempos observados (segundos):')
    totals = Counter()
    for record in times:
        totals[record['stage']] += record['seconds']
    for name, seconds in totals.items():
        print(f'  {name}: {seconds:.1f}')
    print('No extrapolar solo desde ajuste: incluir generación, evaluación, carga y guardado.')
    print('Revisar los tiempos antes de decidir la escala final de 48 ajustes.')
    archive = shutil.make_archive(str(run)+'_resultados', 'zip', root_dir=run)
    print('Paquete:', archive)
    return archive


In [ ]:
ARCHIVE = summarize_pilot(RUN_DIR, RESULTS)


In [ ]:
# Descarga al equipo; no monta Drive ni comparte resultados con terceros.
try:
    from google.colab import files
except ImportError:
    print("Fuera de Colab: el ZIP está en", ARCHIVE)
else:
    files.download(ARCHIVE)


## Qué revisar al terminar

1. `manifest.json` debe indicar `status: complete`, `fit_count: 5` y `test_used: false`.
2. Cada `fit.json` debe mostrar ocho actualizaciones, 8192 tokens objetivo y el mismo hash
   inicial. Un optimizador empieza vacío en cada ajuste.
3. Revisar `short_texts`, truncamientos y textos generados antes de interpretar métricas.
4. Sumar los tiempos, especialmente generación autorregresiva. Decidir escala final con
   margen para interrupciones y redacción. El estudio propuesto de 48 ajustes no se inicia aquí.
5. Conservar el notebook ejecutado y el ZIP. Repetibilidad de semillas no garantiza identidad
   numérica entre versiones, hardware o backends diferentes. El determinismo estricto puede
   detener el piloto si un kernel no lo admite; registrar ese error antes de alterar la política.

### Fuentes de recursos y APIs

- [OPT-125m y sus condiciones de uso](https://huggingface.co/facebook/opt-125m).
- [WikiText, particiones y licencias](https://huggingface.co/datasets/Salesforce/wikitext).
- [MiniLM: mean pooling y normalización](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).
- [Auto classes de Transformers](https://huggingface.co/docs/transformers/main/en/model_doc/auto).
- [Parámetros de generación](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation).
- [Transformers 5.22.2](https://pypi.org/project/transformers/5.22.2/) y
  [Datasets 5.1.0](https://pypi.org/project/datasets/5.1.0/).

Se consultaron estas páginas el 8 de noviembre de 2026 para preparar el notebook.
No se incorporaron referencias nuevas al artículo. El código no redistribuye los pesos
ni el corpus: se descargan al ejecutar y deben respetarse sus condiciones de uso.
